# Tome C2, Chapter 7: Linear Programming (Python companion)

Exercises first, solutions at the end. We implement the simplex method exactly as in the chapter, with dictionaries and exact fractions, Dantzig's and Bland's rules and the two-phase method, and check it against `scipy.optimize.linprog` on random programs. Then we watch a textbook example cycle, count pivots on the Klee–Minty cubes, read dual solutions and Farkas certificates, test total unimodularity by brute force, and finally solve a matroid intersection problem as a linear program with $2^{|E|}$ constraints and check Edmonds' theorem.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, itertools, math, time
from fractions import Fraction as Fr
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog
random.seed(7)

## Exercises

**P7.1 (★★) The simplex method.** Implement dictionaries and pivots (Proposition 7.4) with exact fractions, the rules of Dantzig and Bland (Theorem 7.5) and the two-phase method (Theorem 7.6). Solve the bakery, then compare with `linprog` on a few hundred random programs, some infeasible and some unbounded.

**P7.2 (★) Cycling.** Run Dantzig's rule (ties in the ratio test broken by the smallest index) on Chvátal's example: maximise $10x_1-57x_2-9x_3-24x_4$ subject to $\tfrac12x_1-\tfrac{11}2x_2-\tfrac52x_3+9x_4\le0$, $\tfrac12x_1-\tfrac32x_2-\tfrac12x_3+x_4\le0$, $x_1\le1$, $x\ge0$. Then run Bland's rule.

**P7.3 (★) Klee–Minty.** Count the pivots of both rules on the Klee–Minty cubes of Section 7.7 for $n\le12$, and guess formulas.

**P7.4 (★★) Certificates.** Read the dual solution off the final dictionary (Theorem 7.7), check complementary slackness (Corollary 7.8), and compute Farkas certificates (Corollary 7.9) for random infeasible systems.

**P7.5 (★★) Total unimodularity.** Write a brute-force test of total unimodularity; apply it to incidence matrices of bipartite graphs, of digraphs, of odd cycles, and to interval matrices (Exercise 7.8). Solve the matching linear program on random bipartite and non-bipartite graphs and look at the vertices returned.

**P7.6 (★★★) Matroid intersection by linear programming.** On a small coloured graph, a *rainbow forest* is a forest with edges of distinct colours: a common independent set of the graphic matroid and of the partition matroid of colours. Write the linear program of Theorem 7.21 with all its $2\cdot2^{|E|}$ rank constraints, solve it, and check that the optimal vertex is integral and that its value matches brute force and Edmonds' min–max formula.

## Solutions

### P7.1 The simplex method
A dictionary stores the basic indices `B`, the nonbasic indices `N`, and the coefficients of $x_i=\beta_i-\sum_j\alpha_{ij}x_j$ and $z=\zeta+\sum_j\gamma_jx_j$. The variables are numbered as in the chapter: $x_1,\dots,x_n$, then the slacks $x_{n+1},\dots,x_{n+m}$, and $x_0$ for the auxiliary variable of phase one. In a pivot, the row of the leaving variable is solved for the entering one, and the result, a vector `r` in which position `e` now holds the coefficient of the leaving variable, is substituted everywhere.

In [ ]:
class Dictionary:
    def __init__(self, B, N, beta, alpha, zeta, gamma):
        self.B, self.N = list(B), list(N)
        self.beta, self.alpha = list(beta), [list(r) for r in alpha]
        self.zeta, self.gamma = zeta, list(gamma)

    @classmethod
    def initial(cls, A, b, c):
        m, n = len(A), len(A[0])
        return cls(range(n + 1, n + m + 1), range(1, n + 1), [Fr(x) for x in b],
                   [[Fr(x) for x in row] for row in A], Fr(0), [Fr(x) for x in c])

    def pivot(self, e, l):
        a = self.alpha[l][e]
        r = [x / a for x in self.alpha[l]]; r[e] = 1 / a; bl = self.beta[l] / a
        for i in range(len(self.B)):
            f = self.alpha[i][e]
            if i != l and f:
                self.alpha[i][e] = 0
                self.alpha[i] = [x - f * y for x, y in zip(self.alpha[i], r)]
                self.beta[i] -= f * bl
        g = self.gamma[e]; self.gamma[e] = 0
        self.gamma = [x - g * y for x, y in zip(self.gamma, r)]; self.zeta += g * bl
        self.alpha[l], self.beta[l] = r, bl
        self.B[l], self.N[e] = self.N[e], self.B[l]

    def entering(self, rule):
        cand = [k for k in range(len(self.N)) if self.gamma[k] > 0]
        if not cand: return None
        if rule == "bland": return min(cand, key=lambda k: self.N[k])
        return min(cand, key=lambda k: (-self.gamma[k], self.N[k]))      # Dantzig

    def leaving(self, e):
        rows = [i for i in range(len(self.B)) if self.alpha[i][e] > 0]
        if not rows: return None
        return min(rows, key=lambda i: (self.beta[i] / self.alpha[i][e], self.B[i]))

    def run(self, rule="bland", max_pivots=10**6, trace=None):
        pivots = 0
        while pivots < max_pivots:
            e = self.entering(rule)
            if e is None: return "optimal", pivots
            l = self.leaving(e)
            if l is None: self.ray = e; return "unbounded", pivots
            if trace is not None: trace.append((self.N[e], self.B[l], self.zeta))
            self.pivot(e, l); pivots += 1
        return "stopped", pivots

    def value(self, k):
        return self.beta[self.B.index(k)] if k in self.B else Fr(0)

    def show(self):
        def term(a, k):
            return f" {'-' if a > 0 else '+'} {abs(a)} x{k}" if a else ""
        for i, k in enumerate(self.B):
            print(f"x{k} = {self.beta[i]}" + "".join(term(a, j) for a, j in zip(self.alpha[i], self.N)))
        print(f"z  = {self.zeta}" + "".join(term(-g, j) for g, j in zip(self.gamma, self.N)))

bakery = ([[6, 4], [1, 2], [-1, 1], [0, 1]], [24, 6, 1, 2], [5, 4])
D = Dictionary.initial(*bakery); tr = []
print(D.run("dantzig", trace=tr), "pivots (entering, leaving, z before):", tr)
D.show()

The final dictionary is the one printed in Section 7.2. Now the two-phase method of Theorem 7.6. Phase one adds $x_0$ to every constraint, makes one repairing pivot, and maximises $-x_0$. If $x_0$ is still basic at value $0$, a degenerate pivot removes it. Then the column of $x_0$ is deleted and the true objective is rewritten in terms of the nonbasic variables.

In [ ]:
def two_phase(A, b, c, rule="bland"):
    m, n = len(A), len(A[0])
    if min(b) >= 0:
        D = Dictionary.initial(A, b, c); pivots = 0
    else:
        D = Dictionary(range(n + 1, n + m + 1), [0] + list(range(1, n + 1)), [Fr(x) for x in b],
                       [[Fr(-1)] + [Fr(x) for x in row] for row in A], Fr(0), [Fr(-1)] + [Fr(0)] * n)
        D.pivot(0, min(range(m), key=lambda i: b[i]))
        status, pivots = D.run(rule); pivots += 1
        if D.zeta < 0:
            return "infeasible", None, None, None, pivots
        if 0 in D.B:
            i = D.B.index(0)
            e = next(k for k in range(len(D.N)) if D.alpha[i][k] != 0)
            D.pivot(e, i); pivots += 1
        k0 = D.N.index(0)
        del D.N[k0]
        for row in D.alpha: del row[k0]
        # rewrite z = c.x in terms of the nonbasic variables
        D.zeta, D.gamma = Fr(0), [Fr(0)] * n
        for j in range(1, n + 1):
            cj = Fr(c[j - 1])
            if j in D.N:
                D.gamma[D.N.index(j)] += cj
            else:
                i = D.B.index(j); D.zeta += cj * D.beta[i]
                D.gamma = [g - cj * a for g, a in zip(D.gamma, D.alpha[i])]
    status, p2 = D.run(rule)
    pivots += p2
    if status == "unbounded":
        # certificate: the basic solution x and a direction d with x + t d feasible for all t >= 0
        e = D.ray; x = [D.value(j) for j in range(1, n + 1)]
        d = [Fr(1) if j == D.N[e] else (-D.alpha[D.B.index(j)][e] if j in D.B else Fr(0)) for j in range(1, n + 1)]
        return "unbounded", x, d, None, pivots
    x = [D.value(j) for j in range(1, n + 1)]
    y = [-D.gamma[D.N.index(n + i)] if n + i in D.N else Fr(0) for i in range(1, m + 1)]
    return "optimal", x, y, D.zeta, pivots

print(two_phase(*bakery))
# a program with b not >= 0: maximise x1 + x2 with x1 + x2 >= 2 written as -x1 - x2 <= -2, x1 <= 3, x2 <= 1
print(two_phase([[-1, -1], [1, 0], [0, 1]], [-2, 3, 1], [1, 1]))
print(two_phase([[1, 1], [-1, -1]], [1, -2], [1, 0]))       # infeasible
print(two_phase([[1, -1]], [1], [1, 1]))                     # unbounded

Against `scipy` (HiGHS), on random programs with small integer data, about half with some negative $b_i$. `linprog` minimises, so we pass $-c$; its status codes are 0 (optimal), 2 (infeasible) and 3 (unbounded). Our answers come with exact certificates, which we check: for "optimal", a primal and a dual solution of equal value; for "unbounded", a feasible point $x$ and a direction $d\ge0$ with $Ad\le0$ and $c^{\mathsf T}d>0$. HiGHS gives floating-point answers, and we count the disagreements, if any.

In [ ]:
codes = {0: "optimal", 2: "infeasible", 3: "unbounded"}
tally = {}; agree = certified = 0; trials = 300; disagreements = []
for t in range(trials):
    m, n = random.randint(1, 6), random.randint(1, 6)
    A = [[random.randint(-4, 6) for _ in range(n)] for _ in range(m)]
    b = [random.randint(-3, 10) for _ in range(m)]
    c = [random.randint(-3, 6) for _ in range(n)]
    st, x, y, val, piv = two_phase(A, b, c)
    res = linprog([-v for v in c], A_ub=A, b_ub=b, bounds=[(0, None)] * n, method="highs")
    agree += codes.get(res.status) == st and (st != "optimal" or abs(float(val) + res.fun) < 1e-7)
    if codes.get(res.status) != st: disagreements.append((A, b, c, st, res.status))
    ok = True
    if st == "optimal":
        # x primal feasible, y dual feasible, equal values: an exact proof of optimality
        ok &= all(sum(A[i][j] * x[j] for j in range(n)) <= b[i] for i in range(m)) and min(x) >= 0
        ok &= all(sum(A[i][j] * y[i] for i in range(m)) >= c[j] for j in range(n)) and min(y) >= 0
        ok &= sum(bi * yi for bi, yi in zip(b, y)) == val
    if st == "unbounded":
        d = y
        ok &= min(x) >= 0 and all(sum(A[i][j] * x[j] for j in range(n)) <= b[i] for i in range(m))
        ok &= min(d) >= 0 and all(sum(A[i][j] * d[j] for j in range(n)) <= 0 for i in range(m))
        ok &= sum(cj * dj for cj, dj in zip(c, d)) > 0
    certified += ok and st != "infeasible"; tally[st] = tally.get(st, 0) + 1
print(tally, f"  exact certificates verified: {certified}/{tally['optimal'] + tally['unbounded']},",
      f"  agreement with linprog: {agree}/{trials}")
for A, b, c, st, code in disagreements:
    print("disagreement:", A, b, c, " ours:", st, " HiGHS status:", code)

On the two disagreements (with this seed), HiGHS reports "infeasible", yet in the second one $b\ge0$ so $x=0$ is plainly feasible, and our certificates exhibit a feasible point and an improving ray in both. The culprit is HiGHS's presolve phase: with `options={"presolve": False}` it answers "unbounded" on both. A floating-point solver can be wrong about the status; a checked certificate cannot.

### P7.2 Cycling
With Dantzig's rule the bases repeat with period six and the objective never moves: the method would loop forever. Bland's rule escapes after a few degenerate pivots and finds the optimum $1$.

In [ ]:
A = [[Fr(1, 2), Fr(-11, 2), Fr(-5, 2), 9], [Fr(1, 2), Fr(-3, 2), Fr(-1, 2), 1], [1, 0, 0, 0]]
b, c = [0, 0, 1], [10, -57, -9, -24]
for rule in ("dantzig", "bland"):
    D = Dictionary.initial(A, b, c); tr = []
    status, piv = D.run(rule, max_pivots=14, trace=tr)
    print(f"{rule:8s} {status:8s} after {piv:2d} pivots, z = {D.zeta}")
    print("   entering/leaving:", [(e, l) for e, l, z in tr])

### P7.3 Klee–Minty

In [ ]:
def klee_minty(n):
    A = [[2 * 10**(i - j) if j < i else (1 if j == i else 0) for j in range(n)] for i in range(n)]
    return A, [100**i for i in range(n)], [10**(n - 1 - j) for j in range(n)]

rows = []
for n in range(1, 13):
    counts = []
    for rule in ("dantzig", "bland"):
        D = Dictionary.initial(*klee_minty(n)); status, piv = D.run(rule)
        assert status == "optimal" and D.zeta == 100**(n - 1)
        counts.append(piv)
    rows.append((n, *counts))
    print(f"n={n:2d}  Dantzig {counts[0]:5d}   Bland {counts[1]:4d}")
ns, dz, bl = np.array(rows).T
plt.figure(figsize=(6, 3.3))
plt.semilogy(ns, dz, "o-", label="Dantzig's rule"); plt.semilogy(ns, bl, "s-", label="Bland's rule")
plt.semilogy(ns, 2.0**ns, "k:", lw=1, label="$2^n$")
plt.xlabel("n"); plt.ylabel("pivots"); plt.legend(); plt.tight_layout(); plt.show()

Dantzig's rule makes exactly $2^n-1$ pivots, visiting every vertex of the squashed cube. Bland's rule is faster here, but its counts $1,3,5,9,15,25,41,67,\dots$ satisfy $a_n=a_{n-1}+a_{n-2}+1$, so $a_n=2F_{n+1}-1$ with the Fibonacci numbers: still exponential, with ratio the golden ratio $\approx1.618$. The optimum is always $100^{n-1}$, at the vertex $(0,\dots,0,100^{n-1})$.

In [ ]:
F = [0, 1]
for _ in range(20): F.append(F[-1] + F[-2])
print(all(int(a) == 2 * F[int(n) + 1] - 1 for n, a in zip(ns, bl)))

### P7.4 Certificates
`two_phase` already returns the dual solution $y_i=-\gamma_{n+i}$, and the random test above checked it exactly. Here we check complementary slackness: $y_i>0$ only for tight constraints, and $x_j>0$ only for tight dual constraints.

In [ ]:
viol = 0; count = 0
for t in range(200):
    m, n = random.randint(2, 6), random.randint(2, 6)
    A = [[random.randint(0, 6) for _ in range(n)] for _ in range(m)]
    b = [random.randint(1, 12) for _ in range(m)]; c = [random.randint(0, 6) for _ in range(n)]
    st, x, y, val, _ = two_phase(A, b, c)
    if st != "optimal": continue
    count += 1
    for i in range(m):
        if y[i] > 0 and sum(A[i][j] * x[j] for j in range(n)) != b[i]: viol += 1
    for j in range(n):
        if x[j] > 0 and sum(A[i][j] * y[i] for i in range(m)) != c[j]: viol += 1
print(f"{count} optimal programs, {viol} violations of complementary slackness")

For Farkas' lemma, the proof of Corollary 7.9 says: when $Ax\le b$, $x\ge0$ has no solution, the program "minimise $b^{\mathsf T}y$ subject to $A^{\mathsf T}y\ge0$, $y\ge0$" is unbounded below. Adding the normalisation $\sum_iy_i\le1$ makes it bounded, and its optimum is a certificate $y$ with $b^{\mathsf T}y<0$. We solve it with our own simplex in standard form.

In [ ]:
def farkas(A, b):
    m, n = len(A), len(A[0])
    # maximise -b.y  s.t.  -A^T y <= 0,  sum y <= 1,  y >= 0
    At = [[-A[i][j] for i in range(m)] for j in range(n)] + [[1] * m]
    st, y, _, val, _ = two_phase(At, [0] * n + [1], [-v for v in b])
    return y if val > 0 else None

found = tested = 0
for t in range(300):
    m, n = random.randint(2, 5), random.randint(1, 4)
    A = [[random.randint(-3, 3) for _ in range(n)] for _ in range(m)]
    b = [random.randint(-4, 4) for _ in range(m)]
    if two_phase(A, b, [0] * n)[0] != "infeasible": continue
    tested += 1; y = farkas(A, b)
    assert y is not None and min(y) >= 0 and sum(bi * yi for bi, yi in zip(b, y)) < 0
    assert all(sum(A[i][j] * y[i] for i in range(m)) >= 0 for j in range(n))
    found += 1; example = (A, b, y)
print(f"{tested} infeasible systems, {found} exact Farkas certificates")
A, b, y = example
print("last example: A =", A, " b =", b, "\ncertificate y =", [str(v) for v in y], " b.y =", sum(bi * yi for bi, yi in zip(b, y)))

### P7.5 Total unimodularity
The brute-force test computes every square subdeterminant, exactly, with integer arithmetic (Bareiss would be faster; for these sizes plain fraction elimination is enough).

In [ ]:
def det(M):
    M = [[Fr(x) for x in row] for row in M]; k = len(M); d = Fr(1)
    for col in range(k):
        piv = next((r for r in range(col, k) if M[r][col] != 0), None)
        if piv is None: return 0
        if piv != col: M[col], M[piv] = M[piv], M[col]; d = -d
        d *= M[col][col]
        for r in range(col + 1, k):
            f = M[r][col] / M[col][col]
            M[r] = [a - f * b_ for a, b_ in zip(M[r], M[col])]
    return int(d)

def is_tu(M):
    m, n = len(M), len(M[0])
    for k in range(1, min(m, n) + 1):
        for R in itertools.combinations(range(m), k):
            for Cc in itertools.combinations(range(n), k):
                if det([[M[i][j] for j in Cc] for i in R]) not in (-1, 0, 1):
                    return False, (R, Cc)
    return True, None

def incidence(nv, edges, directed=False):
    M = [[0] * len(edges) for _ in range(nv)]
    for k, (u, v) in enumerate(edges):
        M[u][k] = -1 if directed else 1; M[v][k] = 1
    return M

bip = [(0, 3), (0, 4), (1, 3), (1, 5), (2, 4), (2, 5), (0, 5)]
dig = [(0, 1), (1, 2), (2, 0), (0, 3), (3, 2), (1, 3)]
c5 = [(i, (i + 1) % 5) for i in range(5)]
c6 = [(i, (i + 1) % 6) for i in range(6)]
interval = [[1 if a <= j <= b_ else 0 for j in range(6)] for a, b_ in [(0, 2), (1, 4), (3, 5), (2, 3), (0, 5), (4, 4)]]
for name, M in [("bipartite incidence", incidence(6, bip)), ("digraph incidence", incidence(4, dig, True)),
                ("5-cycle incidence", incidence(5, c5)), ("6-cycle incidence", incidence(6, c6)),
                ("interval matrix", interval)]:
    ok, wit = is_tu(M)
    print(f"{name:20s} TU: {ok}" + ("" if ok else f"   bad submatrix rows {wit[0]} cols {wit[1]}"))

The 5-cycle fails on the whole matrix ($\det=\pm2$), exactly as in Exercise 7.9 for the triangle. Now the matching linear program, solved by the dual simplex of HiGHS, which returns a vertex. On bipartite graphs the vertex is always a matching; on general graphs halves appear, on odd cycles.

In [ ]:
def matching_lp(nv, edges):
    M = incidence(nv, edges)
    res = linprog(-np.ones(len(edges)), A_ub=M, b_ub=np.ones(nv), bounds=[(0, None)] * len(edges), method="highs-ds")
    return res.x, -res.fun

frac = {"bipartite": 0, "general": 0}; T = 200
for t in range(T):
    n = 10
    for kind in frac:
        if kind == "bipartite":
            edges = [(u, v) for u in range(5) for v in range(5, 10) if random.random() < 0.35]
        else:
            edges = [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < 0.25]
        if not edges: continue
        x, val = matching_lp(n, edges)
        frac[kind] += any(abs(xi - round(xi)) > 1e-9 for xi in x)
print("fractional optimal vertices out of", T, ":", frac)
x, val = matching_lp(5, c5)
print("5-cycle: x =", np.round(x, 3), " value", val, " (largest matching: 2)")

### P7.6 Matroid intersection by linear programming
The ground set is the edge set of a small graph whose edges carry colours. $M_1$ is the graphic matroid, with $r_1(A)=|V|-(\text{number of components of }(V,A))$; $M_2$ is the partition matroid of the colours, with $r_2(A)$ the number of colours present in $A$. We write all $2\cdot2^{|E|}$ constraints $x(A)\le r_k(A)$.

In [ ]:
def graphic_rank(nv, edges, A):
    parent = list(range(nv))
    def find(u):
        while parent[u] != u: parent[u] = parent[parent[u]]; u = parent[u]
        return u
    r = 0
    for k in A:
        u, v = find(edges[k][0]), find(edges[k][1])
        if u != v: parent[u] = v; r += 1
    return r

def instance(nv, ne, ncol, seed):
    rng = random.Random(seed)
    pairs = [(u, v) for u in range(nv) for v in range(u + 1, nv)]
    edges = rng.sample(pairs, ne); colours = [rng.randrange(ncol) for _ in edges]
    return edges, colours

def solve_intersection(nv, edges, colours):
    E = len(edges); subsets = [S for k in range(E + 1) for S in itertools.combinations(range(E), k)]
    r1 = {S: graphic_rank(nv, edges, S) for S in subsets}
    r2 = {S: len({colours[k] for k in S}) for S in subsets}
    rows, rhs = [], []
    for r in (r1, r2):
        for S in subsets[1:]:
            rows.append([1 if k in S else 0 for k in range(E)]); rhs.append(r[S])
    res = linprog(-np.ones(E), A_ub=rows, b_ub=rhs, bounds=[(0, None)] * E, method="highs-ds")
    full = tuple(range(E))
    brute = max(len(S) for S in subsets if r1[S] == len(S) and r2[S] == len(S))
    minmax = min(r1[S] + r2[tuple(k for k in full if k not in S)] for S in subsets)
    return res.x, -res.fun, brute, minmax, len(rows)

t0 = time.time(); bad = 0
for seed in range(40):
    edges, colours = instance(6, 9, 4, seed)
    x, lp, brute, minmax, ncons = solve_intersection(6, edges, colours)
    integral = all(abs(xi - round(xi)) < 1e-9 for xi in x)
    bad += not (integral and abs(lp - brute) < 1e-9 and brute == minmax)
print(f"40 instances with |E|=9 ({ncons} constraints each): {bad} disagreements, {time.time() - t0:.1f}s")

Every optimal vertex is integral, the LP value equals the largest rainbow forest, and both equal $\min_A r_1(A)+r_2(E\setminus A)$, as Theorems 7.20 and 7.21 promise. Let us draw one instance with its optimal rainbow forest.

In [ ]:
edges, colours = instance(7, 11, 4, 3)
x, lp, brute, minmax, ncons = solve_intersection(7, edges, colours)
ang = [2 * math.pi * i / 7 for i in range(7)]; pos = [(math.cos(a), math.sin(a)) for a in ang]
palette = ["#1f4e79", "#d9822b", "#a4262c", "#2e7d32"]
plt.figure(figsize=(4.2, 4.2))
for k, (u, v) in enumerate(edges):
    on = x[k] > 0.5
    plt.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color=palette[colours[k]],
             lw=4 if on else 1, ls="-" if on else ":")
plt.scatter(*zip(*pos), s=120, color="white", edgecolor="black", zorder=3)
plt.title(f"largest rainbow forest: {int(round(lp))} edges (thick)", fontsize=10)
plt.axis("off"); plt.axis("equal"); plt.tight_layout(); plt.show()
print("LP value", lp, " brute force", brute, " min-max", minmax)

The linear program here has $2\cdot2^{11}-2=4094$ constraints for $11$ elements: fine for a demonstration, hopeless at scale. Edmonds' exchange-graph algorithm (notebook of Tome C1, Chapter 10) or the ellipsoid method with a separation oracle are the ways to go further.